# Multi-armed bandits

_Classical ML_

**Try arms to learn, but keep pulling the winner. Explore vs exploit.**

A multi-armed bandit is a row of slot machines (arms), each with an unknown payout.

     Every pull gives a reward and a clue about that arm.

---

This notebook is a practice scaffold for the **Multi-armed bandits** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — scikit-learn

We'll implement the **UCB1** bandit algorithm from scratch and run it on three arms with hidden win rates. We build it in three steps: (1) set up the arms and pull each one once, (2) the explore-vs-exploit loop, (3) read off how well it did.

### Step 1 — Set up the arms and seed one pull each

Each arm pays out `1` (a win) with some hidden probability and `0` otherwise. We track two running totals per arm: `sums` (wins so far) and `counts` (pulls so far). Before the algorithm can compare arms it needs at least one sample from each, so we begin by pulling every arm exactly once.

In [ ]:
rng = np.random.default_rng(0)

true_p = np.array([0.3, 0.55, 0.7])   # hidden win rates, unknown to the algorithm
K = len(true_p)                       # number of arms
T = 2000                              # total number of pulls (the budget)

sums = np.zeros(K)     # total reward collected from each arm
counts = np.zeros(K)   # number of times each arm has been pulled

# Seed: pull each arm once so every arm has at least one observation.
for i in range(K):
    reward = rng.random() < true_p[i]   # True/False -> counts as 1/0
    sums[i] += reward
    counts[i] += 1

### Step 2 — Run the UCB1 explore-vs-exploit loop

UCB1 picks the arm with the highest **upper confidence bound**: its estimated mean reward *plus* an optimism bonus. The bonus `sqrt(2·ln(t)/counts)` is large for arms we've pulled rarely, so they get explored, and shrinks as an arm is pulled more, so we eventually settle on (exploit) the arm that looks best. We pull, observe a reward, and fold it into that arm's running totals.

In [ ]:
for t in range(K, T):
    mean = sums / counts                       # current estimated win rate per arm
    bonus = np.sqrt(2 * np.log(t) / counts)    # optimism bonus: bigger for rarely-pulled arms
    ucb = mean + bonus                         # upper confidence bound per arm
    arm = int(np.argmax(ucb))                  # UCB1 picks the arm with the highest bound

    reward = rng.random() < true_p[arm]        # pull it and observe the reward
    sums[arm] += reward
    counts[arm] += 1

### Step 3 — Read off how well it did

Now we compare the algorithm's behaviour against the truth. A good bandit spends most of its budget on the genuinely best arm and recovers estimated means close to the hidden `true_p`. The average reward per pull should sit just below the best arm's true win rate — the small gap is the price of having to explore.

In [ ]:
best = int(np.argmax(true_p))
estimated_means = sums / counts

print("pulls per arm   :", counts.astype(int).tolist())
print("estimated means :", np.round(estimated_means, 3).tolist())
print("true best arm   :", best,
      " pulled %.1f%% of the time" % (100 * counts[best] / T))

total_reward = sums.sum()
avg_reward = total_reward / T
print("avg reward/pull :", round(avg_reward, 3),
      " (best possible %.3f)" % true_p[best])

## Visualize the data & results

_Across three ad creatives, which wins and does UCB beat random rotation?_

### Step 1 — Define the ad-serving simulation

We reframe the bandit as an ad-serving problem: three creatives with realistic click-through rates. The `run` helper plays `T` impressions under a chosen strategy — either `ucb` (the same upper-confidence-bound rule as above) or `random` rotation — and records the **cumulative clicks** after each impression so we can compare the two curves.

In [ ]:
ads = ["Ad A: blue banner", "Ad B: video", "Ad C: carousel"]
true_ctr = np.array([0.06, 0.10, 0.16])   # real-looking click-through rates
K = len(true_ctr)
T = 2000

def run(strategy, seed):
    rng = np.random.default_rng(seed)
    sums = np.zeros(K)
    counts = np.zeros(K)
    cum = []          # cumulative clicks after each impression
    total = 0.0
    for t in range(T):
        if t < K:
            arm = t                       # seed: serve each ad once
        elif strategy == "ucb":
            mean = sums / counts
            bonus = np.sqrt(2 * np.log(t) / counts)
            arm = int(np.argmax(mean + bonus))
        else:
            arm = int(rng.integers(K))    # random rotation
        reward = float(rng.random() < true_ctr[arm])
        sums[arm] += reward
        counts[arm] += 1
        total += reward
        cum.append(total)
    return np.array(cum), counts.astype(int)

### Step 2 — Run both strategies and plot the comparison

We run UCB1 and random rotation, then plot two views. Left: cumulative clicks over time — UCB1 should pull ahead as it learns to favour the best ad. Right: how many impressions UCB1 served to each ad — it should concentrate on the highest-CTR creative (the carousel).

In [ ]:
ucb_cum, ucb_counts = run("ucb", 0)
rnd_cum, _ = run("random", 1)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

# Left — cumulative clicks over time, UCB vs random
ax[0].plot(np.arange(T), ucb_cum, c="#7ee787", label="UCB1")
ax[0].plot(np.arange(T), rnd_cum, c="#ff7b72", label="random")
ax[0].set_title("Cumulative clicks over 2000 impressions: UCB vs random rotation")
ax[0].set_xlabel("impression")
ax[0].set_ylabel("cumulative clicks")
ax[0].legend()

# Right — impressions served per ad by UCB1
ax[1].bar(ads, ucb_counts, color=["#4ea1ff", "#4ea1ff", "#7ee787"])
ax[1].set_title("Impressions served per ad by UCB1")
ax[1].tick_params(axis="x", rotation=20)

plt.show()